# Phase 8 — Qwen Evaluation and Quantization Benchmark

**Archived Kaggle research notebook.** Paths refer to the original Kaggle environment. Review path variables before execution. Some cleanup cells intentionally remove large temporary artifacts under `/kaggle/working`; run those cells only when the targets have been checked. Large datasets, model weights, adapters, and checkpoints are excluded from this GitHub repository.

In [ ]:
!pip install -q --upgrade --no-cache-dir \
    "transformers==4.57.3" \
    "peft==0.19.1" \
    "bitsandbytes==0.50.1" \
    "accelerate==1.14.0" \
    "safetensors==0.8.0"

In [ ]:
import sys
import torch

print("=" * 80)
print("PHASE 8 — CLEAN ENVIRONMENT CHECK")
print("=" * 80)

print("\n========== PYTHON ==========")
print(sys.version)

print("\n========== PYTORCH ==========")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\n========== TRANSFORMERS ==========")
import transformers
print("Transformers:", transformers.__version__)

print("\n========== PEFT ==========")
import peft
print("PEFT:", peft.__version__)

print("\n========== BITSANDBYTES ==========")
import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

print("\n========== ACCELERATE ==========")
import accelerate
print("Accelerate:", accelerate.__version__)

print("\n========== SAFETENSORS ==========")
import safetensors
print("safetensors:", safetensors.__version__)

print("\n" + "=" * 80)
print("ENVIRONMENT CHECK COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 8 — DOWNLOAD BASE QWEN MODEL
# ============================================================

import os
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = "/kaggle/working/qwen2.5-3b-instruct"

print("=" * 80)
print("PHASE 8 — DOWNLOADING QWEN2.5-3B-INSTRUCT")
print("=" * 80)

if os.path.exists(BASE_MODEL_DIR):
    print(f"\nModel directory already exists:")
    print(BASE_MODEL_DIR)
else:
    print("\nDownloading Qwen/Qwen2.5-3B-Instruct...")
    
    snapshot_download(
        repo_id="Qwen/Qwen2.5-3B-Instruct",
        local_dir=BASE_MODEL_DIR,
        local_dir_use_symlinks=False
    )

print("\n" + "=" * 80)
print("DOWNLOAD COMPLETE")
print("=" * 80)

print("Path:", BASE_MODEL_DIR)

print("\nFiles:")
for root, dirs, files in os.walk(BASE_MODEL_DIR):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"{os.path.relpath(path, BASE_MODEL_DIR):70s} {size_mb:8.2f} MB")

In [ ]:
# ============================================================
# VERIFY BASE MODEL
# ============================================================

import os

required_files = [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

print("=" * 80)
print("VERIFYING QWEN BASE MODEL")
print("=" * 80)

for filename in required_files:
    path = os.path.join(BASE_MODEL_DIR, filename)
    print(f"{filename:30s} -> {os.path.exists(path)}")

model_files = []

for root, dirs, files in os.walk(BASE_MODEL_DIR):
    for f in files:
        if f.endswith(".safetensors"):
            model_files.append(os.path.join(root, f))

print("\nSafetensors files:")
for path in model_files:
    size_gb = os.path.getsize(path) / (1024**3)
    print(
        f"{os.path.basename(path):50s} "
        f"{size_gb:.2f} GB"
    )

print("\nNumber of model weight files:", len(model_files))

if not model_files:
    raise RuntimeError(
        "No .safetensors model weights found. "
        "The Qwen download is incomplete."
    )

print("\nBASE MODEL READY")

In [ ]:
# ============================================================
# FIND LORA ADAPTERS
# ============================================================

import os

SEARCH_ROOT = "/kaggle/input/datasets/lucky10406"

print("=" * 80)
print("SEARCHING FOR LORA ADAPTERS")
print("=" * 80)

adapter_files = []

for root, dirs, files in os.walk(SEARCH_ROOT):
    for f in files:
        if f == "adapter_model.safetensors":
            path = os.path.join(root, f)
            adapter_files.append(path)

for path in adapter_files:
    size_mb = os.path.getsize(path) / (1024**2)
    print(f"\n{path}")
    print(f"Size: {size_mb:.2f} MB")

print("\n" + "=" * 80)
print("FOUND:", len(adapter_files), "ADAPTER(S)")
print("=" * 80)

In [ ]:
ADAPTER_PATH = "PUT_THE_FINAL_ADAPTER_PATH_HERE"

print("Using adapter:")
print(ADAPTER_PATH)

In [ ]:
ADAPTER_PATH = "/kaggle/input/datasets/lucky10406/qwen2-5-3b-instruct-official/kaggle/working/qlora-adapter-final"

In [ ]:
# ============================================================
# LOAD TOKENIZER
# ============================================================

from transformers import AutoTokenizer

print("=" * 80)
print("LOADING QWEN TOKENIZER")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_DIR,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nTokenizer loaded successfully!")

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocab size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

In [ ]:
# ============================================================
# PHASE 8 — RECREATE MERGED FINE-TUNED MODEL
# ============================================================

import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

print("=" * 80)
print("PHASE 8 — RECREATING MERGED FINE-TUNED MODEL")
print("=" * 80)

print("\nBase model:")
print(BASE_MODEL_DIR)

print("\nAdapter:")
print(ADAPTER_PATH)

print("\nLoading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Base model loaded!")

print("\nLoading LoRA adapter...")

peft_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

print("LoRA adapter loaded!")

print("\nMerging LoRA adapter...")

merged_model = peft_model.merge_and_unload()

print("Merge completed!")

print("\nMerged model ready.")

In [ ]:
# ============================================================
# PHASE 8 — FIX TORCHAO VERSION
# ============================================================

!pip install -q --upgrade "torchao>=0.17.0"

In [ ]:
# ============================================================
# PHASE 8 — ENVIRONMENT CHECK
# ============================================================

import torch
import transformers
import peft
import bitsandbytes as bnb
import torchao
import accelerate
import safetensors

print("=" * 80)
print("PHASE 8 — ENVIRONMENT CHECK")
print("=" * 80)

print("PyTorch:       ", torch.__version__)
print("Transformers:  ", transformers.__version__)
print("PEFT:          ", peft.__version__)
print("bitsandbytes:  ", bnb.__version__)
print("torchao:       ", torchao.__version__)
print("Accelerate:    ", accelerate.__version__)
print("safetensors:   ", safetensors.__version__)

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:  ", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU 0:         ", torch.cuda.get_device_name(0))
    print("GPU count:     ", torch.cuda.device_count())

print("=" * 80)

# Hard check
from packaging import version

assert version.parse(torchao.__version__) > version.parse("0.16.0"), \
    f"torchao is still too old: {torchao.__version__}"

print("torchao version is compatible with PEFT.")
print("Environment is ready.")

In [ ]:
# ============================================================
# PHASE 8 — RECREATE MERGED FINE-TUNED MODEL
# ============================================================

import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_PATH = "/kaggle/working/qwen2.5-3b-instruct"

ADAPTER_PATH = (
    "/kaggle/input/datasets/lucky10406/"
    "qwen2-5-3b-instruct-official/"
    "kaggle/working/qlora-adapter-final"
)

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print("=" * 80)
print("PHASE 8 — RECREATING MERGED FINE-TUNED MODEL")
print("=" * 80)

print("\nBase model:")
print(BASE_MODEL_PATH)

print("\nAdapter:")
print(ADAPTER_PATH)

# ------------------------------------------------------------
# CHECK PATHS
# ------------------------------------------------------------

if not os.path.exists(BASE_MODEL_PATH):
    raise FileNotFoundError(
        f"Base model not found: {BASE_MODEL_PATH}"
    )

if not os.path.exists(ADAPTER_PATH):
    raise FileNotFoundError(
        f"Adapter not found: {ADAPTER_PATH}"
    )

print("\nBase model exists:  ", os.path.exists(BASE_MODEL_PATH))
print("Adapter exists:     ", os.path.exists(ADAPTER_PATH))

# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    use_fast=True
)

print("Tokenizer loaded!")

# ------------------------------------------------------------
# BASE MODEL
# ------------------------------------------------------------

print("\nLoading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Base model loaded!")

# ------------------------------------------------------------
# LOAD LORA
# ------------------------------------------------------------

print("\nLoading LoRA adapter...")

peft_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

print("LoRA adapter loaded!")

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

print("\nMerging LoRA adapter...")

merged_model = peft_model.merge_and_unload()

print("Merge completed!")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

print("\nSaving merged model...")

os.makedirs(MERGED_PATH, exist_ok=True)

merged_model.save_pretrained(
    MERGED_PATH,
    safe_serialization=True,
    max_shard_size="3GB"
)

tokenizer.save_pretrained(MERGED_PATH)

print("\n" + "=" * 80)
print("MERGED MODEL CREATED SUCCESSFULLY")
print("=" * 80)
print("Path:", MERGED_PATH)

print("\nFiles:")

for root, dirs, files in os.walk(MERGED_PATH):
    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{file:45s} {size_mb:10.2f} MB")

print("=" * 80)

In [ ]:
# ============================================================
# PHASE 8 — LOAD MERGED BASELINE
# ============================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print("=" * 80)
print("PHASE 8 — LOADING MERGED BASELINE")
print("=" * 80)

assert os.path.exists(MERGED_PATH), "Merged model not found!"

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MERGED_PATH,
    use_fast=True
)

print("Tokenizer loaded!")

print("\nLoading merged model...")

merged_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

merged_model.eval()

print("Merged model loaded!")

print("\nDevice map:")
print(getattr(merged_model, "hf_device_map", "not available"))

print("\nGPU memory:")
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved = torch.cuda.memory_reserved(i) / 1024**3
    print(
        f"GPU {i}: allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB"
    )

print("=" * 80)
print("BASELINE READY")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 8 — FIND TEST DATA
# ============================================================

import os

SEARCH_ROOT = "/kaggle/input"

for root, dirs, files in os.walk(SEARCH_ROOT):
    for file in files:
        if file.lower() == "test.csv":
            print(os.path.join(root, file))

In [ ]:
# ============================================================
# LOAD TEST DATA
# ============================================================

TEST_PATH = "/kaggle/input/datasets/lucky10406/datavj/test.csv"

test_df = pd.read_csv(TEST_PATH)

print("Test shape:", test_df.shape)
print("\nColumns:")
print(test_df.columns.tolist())

print("\nFirst rows:")
display(test_df.head())

print("\nLabel distribution:")
print(test_df["class_label"].value_counts())

In [ ]:
# ============================================================
# PROMPT FUNCTION
# ============================================================

LABELS = [
    "injured_or_dead_people",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "infrastructure_and_utility_damage",
    "not_humanitarian",
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "requests_or_urgent_needs",
    "missing_or_found_people",
    "other_relevant_information",
]

def format_prompt(tweet):
    return (
        "Classify the following disaster-related tweet into exactly "
        "one of these categories:\n"
        + "\n".join(LABELS)
        + "\n\n"
        f"Tweet: {tweet}\n\n"
        "Answer:"
    )

In [ ]:
# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict(tweet, model, tokenizer):
    prompt = format_prompt(tweet)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Put inputs on the same device as the model's input embeddings
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # Extract one valid label
    # --------------------------------------------------------

    generated_lower = generated_text.lower()

    # Exact label matching
    for label in LABELS:
        if label in generated_lower:
            return label

    return "unmatched"

In [ ]:
# ============================================================
# BASELINE SANITY CHECK
# ============================================================

sample_tweets = test_df["text_clean"].sample(
    5,
    random_state=42
).tolist()

for tweet in sample_tweets:

    prediction = predict(
        tweet,
        merged_model,
        tokenizer
    )

    print("=" * 80)
    print("Tweet:", tweet)
    print("Prediction:", prediction)

In [ ]:
# ============================================================
# CLASSIFICATION BENCHMARK
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

def benchmark_classification(
    model,
    tokenizer,
    test_df,
    model_name,
    limit=None
):
    if limit is not None:
        df = test_df.sample(
            n=min(limit, len(test_df)),
            random_state=42
        ).copy()
    else:
        df = test_df.copy()

    predictions = []

    start = time.perf_counter()

    for i, tweet in enumerate(df["text_clean"]):

        pred = predict(
            tweet,
            model,
            tokenizer
        )

        predictions.append(pred)

        if (i + 1) % 100 == 0:
            print(
                f"{model_name}: "
                f"{i + 1}/{len(df)}"
            )

    total_time = time.perf_counter() - start

    df["predicted"] = predictions

    y_true = df["class_label"]
    y_pred = df["predicted"]

    results = {
        "model": model_name,
        "n_samples": len(df),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0
        ),

        "total_inference_time_s": total_time,

        "samples_per_second": (
            len(df) / total_time
        ),
    }

    return results, df

In [ ]:
# ============================================================
# BASELINE — 100 SAMPLE TEST
# ============================================================

baseline_100_results, baseline_100_predictions = (
    benchmark_classification(
        merged_model,
        tokenizer,
        test_df,
        "baseline_finetuned",
        limit=100
    )
)

print("\n" + "=" * 80)
print("BASELINE 100-SAMPLE RESULTS")
print("=" * 80)

for key, value in baseline_100_results.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# PREDICTION CHECK
# ============================================================

print(
    baseline_100_predictions[
        [
            "text_clean",
            "class_label",
            "predicted"
        ]
    ].head(20).to_string(index=False)
)

In [ ]:
print("\nPrediction distribution:")
print(
    baseline_100_predictions["predicted"]
    .value_counts()
)

In [ ]:
# ============================================================
# LATENCY BENCHMARK
# ============================================================

def measure_latency(
    model,
    tokenizer,
    sample_tweet,
    n_trials=50,
    warmup=5
):

    prompt = format_prompt(sample_tweet)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    input_device = model.get_input_embeddings().weight.device

    inputs = {
        k: v.to(input_device)
        for k, v in inputs.items()
    }

    # --------------------------------------------------------
    # Warmup
    # --------------------------------------------------------

    for _ in range(warmup):

        with torch.no_grad():

            model.generate(
                **inputs,
                max_new_tokens=15,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # --------------------------------------------------------
    # Timed trials
    # --------------------------------------------------------

    times = []

    for _ in range(n_trials):

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.no_grad():

            model.generate(
                **inputs,
                max_new_tokens=15,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        times.append(elapsed)

    return {
        "latency_median_s": float(np.median(times)),
        "latency_mean_s": float(np.mean(times)),
        "latency_std_s": float(np.std(times)),
        "throughput_per_s": float(
            1.0 / np.median(times)
        ),
    }

In [ ]:
# ============================================================
# BASELINE LATENCY
# ============================================================

latency_baseline = measure_latency(
    merged_model,
    tokenizer,
    test_df["text_clean"].iloc[0],
    n_trials=50,
    warmup=5
)

print("=" * 80)
print("BASELINE LATENCY")
print("=" * 80)

for key, value in latency_baseline.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# GPU MEMORY
# ============================================================

def get_gpu_memory():

    memory = {}

    for i in range(torch.cuda.device_count()):

        memory[f"gpu_{i}_allocated_gb"] = (
            torch.cuda.memory_allocated(i)
            / (1024 ** 3)
        )

        memory[f"gpu_{i}_reserved_gb"] = (
            torch.cuda.memory_reserved(i)
            / (1024 ** 3)
        )

        memory[f"gpu_{i}_max_allocated_gb"] = (
            torch.cuda.max_memory_allocated(i)
            / (1024 ** 3)
        )

        memory[f"gpu_{i}_max_reserved_gb"] = (
            torch.cuda.max_memory_reserved(i)
            / (1024 ** 3)
        )

    return memory


torch.cuda.reset_peak_memory_stats()

# One inference
_ = predict(
    test_df["text_clean"].iloc[0],
    merged_model,
    tokenizer
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

baseline_memory = get_gpu_memory()

print("=" * 80)
print("BASELINE GPU MEMORY")
print("=" * 80)

for key, value in baseline_memory.items():
    print(f"{key}: {value:.3f} GB")

In [ ]:
# ============================================================
# MODEL SIZE
# ============================================================

def get_dir_size(path):

    total = 0

    for dirpath, _, filenames in os.walk(path):

        for filename in filenames:

            filepath = os.path.join(
                dirpath,
                filename
            )

            total += os.path.getsize(filepath)

    return total / (1024 ** 3)


baseline_size_gb = get_dir_size(
    MERGED_PATH
)

print(
    f"Baseline model size: "
    f"{baseline_size_gb:.3f} GB"
)

In [ ]:
# ============================================================
# SAVE BASELINE RESULTS
# ============================================================

RESULTS_DIR = "/kaggle/working/phase8-results"

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

baseline_results = {
    **baseline_100_results,
    **latency_baseline,
    **baseline_memory,
    "model_size_gb": baseline_size_gb,
}

with open(
    os.path.join(
        RESULTS_DIR,
        "baseline_results.json"
    ),
    "w"
) as f:

    json.dump(
        baseline_results,
        f,
        indent=2
    )

print("=" * 80)
print("BASELINE RESULTS SAVED")
print("=" * 80)

print(
    os.path.join(
        RESULTS_DIR,
        "baseline_results.json"
    )
)

In [ ]:
# ============================================================
# FREE BASELINE MODEL FROM GPU
# ============================================================

import gc
import torch

print("Freeing baseline model from GPU memory...")

if "merged_model" in globals():
    del merged_model

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

print("GPU memory cleared.")

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated"
    )

In [ ]:
# ============================================================
# LOAD INT8 MODEL
# ============================================================

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print("=" * 80)
print("PHASE 8 — LOADING INT8 MODEL")
print("=" * 80)

int8_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    quantization_config=int8_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model_int8.eval()

print("\nINT8 MODEL LOADED SUCCESSFULLY")

print("\nDevice map:")
print(getattr(model_int8, "hf_device_map", "not available"))

print("\nis_loaded_in_8bit:",
      getattr(model_int8, "is_loaded_in_8bit", "not available"))

print("=" * 80)

In [ ]:
# ============================================================
# INT8 SANITY CHECK
# ============================================================

sample_tweet = test_df["text_clean"].iloc[0]

print("Tweet:")
print(sample_tweet)

print("\nINT8 prediction:")

int8_test_prediction = predict(
    sample_tweet,
    model_int8,
    tokenizer
)

print(int8_test_prediction)

In [ ]:
# ============================================================
# INT8 — SAME 100 SAMPLE BENCHMARK
# ============================================================

int8_100_results, int8_100_predictions = (
    benchmark_classification(
        model_int8,
        tokenizer,
        test_df,
        "int8",
        limit=100
    )
)

print("\n" + "=" * 80)
print("INT8 100-SAMPLE RESULTS")
print("=" * 80)

for key, value in int8_100_results.items():
    print(f"{key}: {value}")

In [ ]:
print("=" * 80)
print("INT8 PREDICTION DISTRIBUTION")
print("=" * 80)

print(
    int8_100_predictions["predicted"]
    .value_counts()
)

print("\nUnmatched:")

print(
    (
        int8_100_predictions["predicted"]
        == "unmatched"
    ).sum()
)

In [ ]:
print(
    int8_100_predictions[
        [
            "text_clean",
            "class_label",
            "predicted"
        ]
    ].head(20).to_string(index=False)
)

In [ ]:
# ============================================================
# INT8 LATENCY
# ============================================================

latency_int8 = measure_latency(
    model_int8,
    tokenizer,
    test_df["text_clean"].iloc[0],
    n_trials=50,
    warmup=5
)

print("=" * 80)
print("INT8 LATENCY")
print("=" * 80)

for key, value in latency_int8.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# INT8 GPU MEMORY
# ============================================================

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

_ = predict(
    test_df["text_clean"].iloc[0],
    model_int8,
    tokenizer
)

torch.cuda.synchronize()

int8_memory = get_gpu_memory()

print("=" * 80)
print("INT8 GPU MEMORY")
print("=" * 80)

for key, value in int8_memory.items():
    print(f"{key}: {value:.3f} GB")

In [ ]:
# ============================================================
# SAVE INT8 RESULTS
# ============================================================

import json
import os

RESULTS_DIR = "/kaggle/working/phase8-results"

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

int8_results = {
    **int8_100_results,
    **latency_int8,
    **int8_memory,
    "quantization": "bitsandbytes_int8",
    "load_in_8bit": True
}

INT8_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "int8_results.json"
)

with open(
    INT8_RESULTS_PATH,
    "w"
) as f:
    json.dump(
        int8_results,
        f,
        indent=2
    )

# Save the exact 100 predictions too
int8_100_predictions.to_csv(
    os.path.join(
        RESULTS_DIR,
        "int8_100_predictions.csv"
    ),
    index=False
)

print("=" * 80)
print("INT8 RESULTS SAVED")
print("=" * 80)

print(INT8_RESULTS_PATH)

In [ ]:
# ============================================================
# BASELINE VS INT8
# ============================================================

BASELINE_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "baseline_results.json"
)

with open(BASELINE_RESULTS_PATH) as f:
    baseline_saved = json.load(f)

comparison = pd.DataFrame([
    {
        "model": "baseline",
        "accuracy": baseline_saved["accuracy"],
        "macro_f1": baseline_saved["macro_f1"],
        "weighted_f1": baseline_saved["weighted_f1"],
        "latency_median_s":
            baseline_saved["latency_median_s"],
    },
    {
        "model": "INT8",
        "accuracy": int8_results["accuracy"],
        "macro_f1": int8_results["macro_f1"],
        "weighted_f1": int8_results["weighted_f1"],
        "latency_median_s":
            int8_results["latency_median_s"],
    }
])

display(comparison)

In [ ]:
print("=" * 80)
print("INT8 CHANGE FROM BASELINE")
print("=" * 80)

print(
    "Accuracy change:",
    int8_results["accuracy"]
    - baseline_saved["accuracy"]
)

print(
    "Macro F1 change:",
    int8_results["macro_f1"]
    - baseline_saved["macro_f1"]
)

print(
    "Weighted F1 change:",
    int8_results["weighted_f1"]
    - baseline_saved["weighted_f1"]
)

print(
    "Latency change:",
    int8_results["latency_median_s"]
    - baseline_saved["latency_median_s"],
    "seconds"
)

In [ ]:
# ============================================================
# FREE INT8 MODEL
# ============================================================

import gc
import torch

if "model_int8" in globals():
    del model_int8

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

print("INT8 model removed from GPU.")

In [ ]:
# ============================================================
# LOAD INT4 MODEL
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print("=" * 80)
print("PHASE 8 — LOADING INT4 MODEL")
print("=" * 80)

int4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    quantization_config=int4_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model_int4.eval()

print("\nINT4 MODEL LOADED SUCCESSFULLY")

print("is_loaded_in_4bit:",
      getattr(model_int4, "is_loaded_in_4bit", "not available"))

print("Device map:")
print(getattr(model_int4, "hf_device_map", "not available"))

In [ ]:
# ============================================================
# INT4 SANITY CHECK
# ============================================================

sample_tweet = test_df["text_clean"].iloc[0]

print("Tweet:")
print(sample_tweet)

print("\nINT4 prediction:")

print(
    predict(
        sample_tweet,
        model_int4,
        tokenizer
    )
)

In [ ]:
# ============================================================
# INT4 — 100 SAMPLE BENCHMARK
# ============================================================

int4_100_results, int4_100_predictions = (
    benchmark_classification(
        model_int4,
        tokenizer,
        test_df,
        "int4",
        limit=100
    )
)

print("\n" + "=" * 80)
print("INT4 100-SAMPLE RESULTS")
print("=" * 80)

for key, value in int4_100_results.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# INT4 LATENCY
# ============================================================

latency_int4 = measure_latency(
    model_int4,
    tokenizer,
    test_df["text_clean"].iloc[0],
    n_trials=50,
    warmup=5
)

print("=" * 80)
print("INT4 LATENCY")
print("=" * 80)

for key, value in latency_int4.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# INT4 GPU MEMORY
# ============================================================

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

_ = predict(
    test_df["text_clean"].iloc[0],
    model_int4,
    tokenizer
)

torch.cuda.synchronize()

int4_memory = get_gpu_memory()

print("=" * 80)
print("INT4 GPU MEMORY")
print("=" * 80)

for key, value in int4_memory.items():
    print(f"{key}: {value:.3f} GB")

In [ ]:
# ============================================================
# SAVE INT4 RESULTS
# ============================================================

import json
import os

RESULTS_DIR = "/kaggle/working/phase8-results"

int4_results = {
    **int4_100_results,
    **latency_int4,
    **int4_memory,
    "quantization": "bitsandbytes_nf4",
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_use_double_quant": True
}

INT4_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "int4_results.json"
)

with open(INT4_RESULTS_PATH, "w") as f:
    json.dump(
        int4_results,
        f,
        indent=2
    )

int4_100_predictions.to_csv(
    os.path.join(
        RESULTS_DIR,
        "int4_100_predictions.csv"
    ),
    index=False
)

print("=" * 80)
print("INT4 RESULTS SAVED")
print("=" * 80)
print(INT4_RESULTS_PATH)

In [ ]:
# ============================================================
# BASELINE vs INT8 vs INT4
# ============================================================

comparison = pd.DataFrame([
    {
        "model": "baseline",
        "accuracy": baseline_saved["accuracy"],
        "macro_f1": baseline_saved["macro_f1"],
        "weighted_f1": baseline_saved["weighted_f1"],
        "latency_median_s": baseline_saved["latency_median_s"],
    },
    {
        "model": "INT8",
        "accuracy": int8_results["accuracy"],
        "macro_f1": int8_results["macro_f1"],
        "weighted_f1": int8_results["weighted_f1"],
        "latency_median_s": int8_results["latency_median_s"],
    },
    {
        "model": "INT4",
        "accuracy": int4_results["accuracy"],
        "macro_f1": int4_results["macro_f1"],
        "weighted_f1": int4_results["weighted_f1"],
        "latency_median_s": int4_results["latency_median_s"],
    }
])

display(comparison)

In [ ]:
# ============================================================
# PHASE 8 — FULL INT4 EVALUATION
# ============================================================

print("=" * 80)
print("FULL INT4 TEST-SET EVALUATION")
print("Samples:", len(test_df))
print("=" * 80)

int4_full_results, int4_full_predictions = (
    benchmark_classification(
        model_int4,
        tokenizer,
        test_df,
        "int4",
        limit=None
    )
)

print("\n" + "=" * 80)
print("INT4 FULL RESULTS")
print("=" * 80)

for key, value in int4_full_results.items():
    print(f"{key}: {value}")

# Save immediately
int4_full_predictions.to_csv(
    "/kaggle/working/phase8-results/int4_full_predictions.csv",
    index=False
)

with open(
    "/kaggle/working/phase8-results/int4_full_results.json",
    "w"
) as f:
    json.dump(int4_full_results, f, indent=2)

print("\nFULL INT4 RESULTS SAVED")

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Keep exactly 2,000 examples with the same class proportions
_, benchmark_df = train_test_split(
    test_df,
    test_size=2000,
    stratify=test_df["class_label"],
    random_state=42
)

benchmark_df = benchmark_df.reset_index(drop=True)

print("Benchmark shape:", benchmark_df.shape)

print("\nClass distribution:")
print(benchmark_df["class_label"].value_counts())

# Save the exact benchmark subset so all models use identical examples
benchmark_df.to_csv(
    "/kaggle/working/phase8-results/benchmark_2000.csv",
    index=False
)

print("\nSaved benchmark subset.")

In [ ]:
int4_full_results, int4_full_predictions = benchmark_classification(
    model_int4,
    tokenizer,
    benchmark_df,
    "int4",
    limit=None
)

print(int4_full_results)

int4_full_predictions.to_csv(
    "/kaggle/working/phase8-results/int4_2000_predictions.csv",
    index=False
)

with open(
    "/kaggle/working/phase8-results/int4_2000_results.json",
    "w"
) as f:
    json.dump(int4_full_results, f, indent=2)

In [ ]:
import gc
import torch

del model_int4
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

int8_config = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    quantization_config=int8_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model_int8.eval()

In [ ]:
int8_full_results, int8_full_predictions = benchmark_classification(
    model_int8,
    tokenizer,
    benchmark_df,
    "int8",
    limit=None
)

print(int8_full_results)

int8_full_predictions.to_csv(
    "/kaggle/working/phase8-results/int8_2000_predictions.csv",
    index=False
)

with open(
    "/kaggle/working/phase8-results/int8_2000_results.json",
    "w"
) as f:
    json.dump(int8_full_results, f, indent=2)

In [ ]:
del model_int8
gc.collect()
torch.cuda.empty_cache()

In [ ]:
merged_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

merged_model.eval()

In [ ]:
baseline_2000_results, baseline_2000_predictions = benchmark_classification(
    merged_model,
    tokenizer,
    benchmark_df,
    "baseline_finetuned",
    limit=None
)

print(baseline_2000_results)

baseline_2000_predictions.to_csv(
    "/kaggle/working/phase8-results/baseline_2000_predictions.csv",
    index=False
)

with open(
    "/kaggle/working/phase8-results/baseline_2000_results.json",
    "w"
) as f:
    json.dump(baseline_2000_results, f, indent=2)

In [ ]:
import json
import os

RESULTS_DIR = "/kaggle/working/phase8-results"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(
    os.path.join(RESULTS_DIR, "baseline_2000_results.json"),
    "w"
) as f:
    json.dump(baseline_2000_results, f, indent=2)

baseline_2000_predictions.to_csv(
    os.path.join(RESULTS_DIR, "baseline_2000_predictions.csv"),
    index=False
)

print("Baseline 2000 results saved.")

In [ ]:
import gc
import torch

if "merged_model" in globals():
    del merged_model

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

print("Baseline model removed from GPU.")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MERGED_PATH = "/kaggle/working/merged-finetuned-model"

print("=" * 80)
print("LOADING INT8")
print("=" * 80)

int8_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    quantization_config=int8_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model_int8.eval()

print("INT8 loaded successfully.")

In [ ]:
# ============================================================
# EMERGENCY PHASE 8 SESSION BACKUP
# Run this BEFORE closing Kaggle
# ============================================================

import os
import json
import shutil
import zipfile
import subprocess
from datetime import datetime

RESULTS_DIR = "/kaggle/working/phase8-results"
BACKUP_DIR = "/kaggle/working/phase8-session-backup"
ZIP_PATH = "/kaggle/working/phase8-session-backup.zip"

os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Save benchmark subset again if it exists
# ------------------------------------------------------------

if "benchmark_df" in globals():
    benchmark_df.to_csv(
        os.path.join(RESULTS_DIR, "benchmark_2000.csv"),
        index=False
    )
    print("✅ benchmark_2000.csv saved")

# ------------------------------------------------------------
# 2. Save important result variables if they currently exist
# ------------------------------------------------------------

json_variables = {
    "baseline_2000_results": "baseline_2000_results.json",
    "int8_2000_results": "int8_2000_results.json",
    "int4_2000_results": "int4_2000_results.json",

    "latency_int8": "int8_latency.json",
    "latency_int4": "int4_latency.json",
    "latency_baseline_2": "baseline_latency.json",

    "int8_memory": "int8_memory.json",
    "int4_memory": "int4_memory.json",
    "baseline_memory_2": "baseline_memory.json",

    "baseline_energy": "baseline_energy.json",
    "int8_energy": "int8_energy.json",
    "int4_energy": "int4_energy.json",
}

for variable_name, filename in json_variables.items():

    if variable_name in globals():

        try:
            with open(
                os.path.join(RESULTS_DIR, filename),
                "w"
            ) as f:

                json.dump(
                    globals()[variable_name],
                    f,
                    indent=2,
                    default=str
                )

            print(f"✅ {filename}")

        except Exception as e:
            print(f"⚠️ Could not save {variable_name}: {e}")


# ------------------------------------------------------------
# 3. Save prediction DataFrames if they exist
# ------------------------------------------------------------

dataframe_variables = {
    "baseline_2000_predictions":
        "baseline_2000_predictions.csv",

    "int8_2000_predictions":
        "int8_2000_predictions.csv",

    "int4_2000_predictions":
        "int4_2000_predictions.csv",

    "int8_100_predictions":
        "int8_100_predictions.csv",

    "int4_100_predictions":
        "int4_100_predictions.csv",
}

for variable_name, filename in dataframe_variables.items():

    if variable_name in globals():

        try:

            globals()[variable_name].to_csv(
                os.path.join(RESULTS_DIR, filename),
                index=False
            )

            print(f"✅ {filename}")

        except Exception as e:
            print(f"⚠️ Could not save {variable_name}: {e}")


# ------------------------------------------------------------
# 4. Save package/environment versions
# ------------------------------------------------------------

packages = [
    "torch",
    "transformers",
    "peft",
    "bitsandbytes",
    "accelerate",
    "torchao",
    "safetensors",
]

environment = {}

for package in packages:

    try:
        module = __import__(package)
        environment[package] = getattr(
            module,
            "__version__",
            "unknown"
        )

    except Exception as e:
        environment[package] = f"not available: {e}"


environment["cuda_available"] = str(
    __import__("torch").cuda.is_available()
)

environment["timestamp"] = datetime.now().isoformat()


with open(
    os.path.join(RESULTS_DIR, "environment.json"),
    "w"
) as f:

    json.dump(
        environment,
        f,
        indent=2
    )

print("✅ environment.json saved")


# ------------------------------------------------------------
# 5. Save recovery instructions
# ------------------------------------------------------------

recovery_text = """
PHASE 8 SESSION RECOVERY
========================

BASE MODEL:
Qwen/Qwen2.5-3B-Instruct

BASE MODEL DOWNLOAD TARGET:
/kaggle/working/qwen2.5-3b-instruct

FINAL QLORA ADAPTER:
/kaggle/input/datasets/lucky10406/qwen2-5-3b-instruct-official/kaggle/working/qlora-adapter-final

TEST DATA:
/kaggle/input/datasets/lucky10406/datavj/test.csv

MERGED MODEL TEMPORARY PATH:
/kaggle/working/merged-finetuned-model

IMPORTANT:
The base model and merged model are intentionally NOT included
in this backup because they are several GB and can be recreated.

PHASE 8 DIRECT COMPARISON:
2000 stratified test examples
random_state = 42

BASELINE 2000 RESULT:
accuracy = 0.741
macro_precision = 0.7503942199009614
macro_recall = 0.7493229252252915
macro_f1 = 0.732505725427883
weighted_f1 = 0.7310994656252274
samples_per_second = 0.9761429319042676

100 SAMPLE SCREENING:
Baseline:
accuracy = 0.68
macro_f1 = 0.683176
weighted_f1 = 0.672102
median latency = 1.019009 s

INT8:
accuracy = 0.67
macro_f1 = 0.677874
weighted_f1 = 0.663847
median latency = 3.218189 s

INT4:
accuracy = 0.71
macro_f1 = 0.701342
weighted_f1 = 0.711423
median latency = 1.301427 s

NEXT:
Continue Phase 8 with INT8 and INT4 on the same benchmark_2000.csv,
then runtime/memory/energy/final metrics.
"""

with open(
    os.path.join(RESULTS_DIR, "SESSION_RECOVERY.txt"),
    "w",
    encoding="utf-8"
) as f:

    f.write(recovery_text)

print("✅ SESSION_RECOVERY.txt saved")


# ------------------------------------------------------------
# 6. Make clean backup directory
# ------------------------------------------------------------

if os.path.exists(BACKUP_DIR):
    shutil.rmtree(BACKUP_DIR)

shutil.copytree(
    RESULTS_DIR,
    BACKUP_DIR
)


# ------------------------------------------------------------
# 7. Create ZIP
# ------------------------------------------------------------

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

shutil.make_archive(
    ZIP_PATH[:-4],
    "zip",
    root_dir=BACKUP_DIR
)


# ------------------------------------------------------------
# 8. Verify
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BACKUP COMPLETE")
print("=" * 80)

print("ZIP:")
print(ZIP_PATH)

print(
    "\nSize:",
    round(os.path.getsize(ZIP_PATH) / 1024**2, 2),
    "MB"
)

print("\nZIP CONTENTS:\n")

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    for name in z.namelist():
        print(" ", name)